In [1]:
import asyncio
import aiohttp
import pandas as pd
import re
import pymorphy3
from nltk.corpus import stopwords

In [5]:
QUERY = "Python"
AREA = 113
MAX_VACANCIES = 50
PAGE_SIZE = 20

morph = pymorphy3.MorphAnalyzer()
russian_stopwords = set(stopwords.words("russian"))

In [6]:
def clean_description(text):
    if not text:
        return ""

    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^а-яёa-z\s]', ' ', text, flags=re.IGNORECASE)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    lemmatized_words = []
    
    for word in words:
        if word not in russian_stopwords and len(word) > 2:
            parsed_word = morph.parse(word)[0]
            lemmatized_words.append(parsed_word.normal_form)
    
    return ' '.join(lemmatized_words)

In [7]:
async def fetch_vacancy_details(session, vacancy_id):
    try:
        url = f"https://api.hh.ru/vacancies/{vacancy_id}"
        async with session.get(url) as response:
            if response.status == 200:
                data = await response.json()
                
                return {
                    'id': vacancy_id,
                    'name': data.get('name', ''),
                    'description': clean_description(data.get('description', '')),
                    'employer': data.get('employer', {}).get('name', ''),
                    'salary_from': data.get('salary', {}).get('from'),
                    'salary_to': data.get('salary', {}).get('to'),
                    'salary_currency': data.get('salary', {}).get('currency'),
                    'experience': data.get('experience', {}).get('name', ''),
                    'employment': data.get('employment', {}).get('name', '')
                }
            else:
                return None
    except Exception as e:
        print(f"Ошибка при получении вакансии {vacancy_id}: {e}")
        return None

In [ ]:
async def search_vacancies():
    all_vacancies = []
    page = 0
    
    async with aiohttp.ClientSession() as session:
        while len(all_vacancies) < MAX_VACANCIES:
            url = "https://api.hh.ru/vacancies"
            params = {
                "text": QUERY,
                "area": AREA,
                "page": page,
                "per_page": PAGE_SIZE
            }
            
            try:
                async with session.get(url, params=params) as response:
                    if response.status == 200:
                        data = await response.json()
                        vacancies = data.get('items', [])
                        
                        if not vacancies:
                            break

                        tasks = []
                        for vacancy in vacancies:
                            if len(all_vacancies) < MAX_VACANCIES:
                                tasks.append(fetch_vacancy_details(session, vacancy['id']))

                        results = await asyncio.gather(*tasks)

                        for result in results:
                            if result:
                                all_vacancies.append(result)
                        
                        print(f"Загружена страница {page + 1}, вакансий: {len(all_vacancies)}")
                        
                        page += 1

                        if page >= data.get('pages', 1):
                            break
                    else:
                        print(f"Ошибка HTTP: {response.status}")
                        break
            except Exception as e:
                print(f"Ошибка при поиске: {e}")
                break
    
    return all_vacancies

In [9]:
vacancies_data = await search_vacancies()
print(f"Получено вакансий: {len(vacancies_data)}")

Ошибка при получении вакансии 127029635: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 127073497: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 126983766: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 127080674: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 127080131: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 126173251: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 127065972: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 127081588: 'NoneType' object has no attribute 'get'
Загружена страница 1, вакансий: 12
Ошибка при получении вакансии 127056133: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 126983451: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 126974149: 'NoneType' object has no attribute 'get'
Ошибка при получении вакансии 127074757: 'NoneType' object has 

In [10]:
df = pd.DataFrame(vacancies_data)

print(f"Размер: {df.shape}")
df.head()

Размер: (55, 9)


,id,name,description,employer,salary_from,salary_to,salary_currency,experience,employment
0,127075950,Ручной тестировщик,привет компания skif приглашать тестировщик оп...,Свистунова Екатерина Александровна,180000.0,230000.0,RUR,Более 6 лет,Полная занятость
1,126973641,Программист (Junior - младший разработчик),наш команда требоваться программист требование...,ПТМК,50000.0,70000.0,RUR,Нет опыта,Полная занятость
2,126443956,Junior\Middle Backend Python разработчик (Fast...,технологический команда внутри один топ экспед...,АЛТ,50000.0,70000.0,RUR,От 1 года до 3 лет,Полная занятость
3,127050528,Аналитик данных,почему стоить присоединиться влияние бизнес тв...,Совкомбанк Страхование,135000.0,135000.0,RUR,От 1 года до 3 лет,Полная занятость
4,126963102,Стажер-исследователь/аналитик,институт статистический исследование экономика...,Национальный исследовательский университет Выс...,50000.0,100000.0,RUR,Нет опыта,Полная занятость


In [11]:
print(f"Всего вакансий: {len(df)}")
print(f"Вакансий с указанной зарплатой: {df['salary_from'].notna().sum()}")
print(f"Уникальных работодателей: {df['employer'].nunique()}")

Всего вакансий: 55
Вакансий с указанной зарплатой: 44
Уникальных работодателей: 54


In [12]:
df.to_csv('hh_vacancies_dataset.csv', index=False, encoding='utf-8')
df.to_json('hh_vacancies_dataset.json', orient='records', force_ascii=False, indent=2)

In [13]:
df_check = pd.read_csv('hh_vacancies_dataset.csv')
print("Проверка загруженных данных:")
print(f"Загружено записей: {len(df_check)}")
print("\nПример вакансии:")
print(f"Название: {df_check.iloc[0]['name']}")
print(f"Работодатель: {df_check.iloc[0]['employer']}")
print(f"Описание (первые 100 символов): {df_check.iloc[0]['description'][:100]}...")

Проверка загруженных данных:
Загружено записей: 55

Пример вакансии:
Название: Ручной тестировщик
Работодатель: Свистунова Екатерина Александровна
Описание (первые 100 символов): привет компания skif приглашать тестировщик опыт год компания разрабатывать продукт мониторинг транс...


p.s. Так как соединение токенов в текст через пробел бесполезно - токенизирую повторно и пересохраняю

In [2]:
df = pd.read_csv('hh_vacancies_dataset.csv')
df['description_tokens'] = df['description'].apply(lambda x: x.split() if pd.notna(x) else [])
df.to_csv('hh_vacancies_dataset.csv', index=False, encoding='utf-8')
df.head()

,id,name,description,employer,salary_from,salary_to,salary_currency,experience,employment,description_tokens
0,127075950,Ручной тестировщик,привет компания skif приглашать тестировщик оп...,Свистунова Екатерина Александровна,180000.0,230000.0,RUR,Более 6 лет,Полная занятость,"[привет, компания, skif, приглашать, тестировщ..."
1,126973641,Программист (Junior - младший разработчик),наш команда требоваться программист требование...,ПТМК,50000.0,70000.0,RUR,Нет опыта,Полная занятость,"[наш, команда, требоваться, программист, требо..."
2,126443956,Junior\Middle Backend Python разработчик (Fast...,технологический команда внутри один топ экспед...,АЛТ,50000.0,70000.0,RUR,От 1 года до 3 лет,Полная занятость,"[технологический, команда, внутри, один, топ, ..."
3,127050528,Аналитик данных,почему стоить присоединиться влияние бизнес тв...,Совкомбанк Страхование,135000.0,135000.0,RUR,От 1 года до 3 лет,Полная занятость,"[почему, стоить, присоединиться, влияние, бизн..."
4,126963102,Стажер-исследователь/аналитик,институт статистический исследование экономика...,Национальный исследовательский университет Выс...,50000.0,100000.0,RUR,Нет опыта,Полная занятость,"[институт, статистический, исследование, эконо..."


In [3]:
import json

data_dict = df.to_dict('records')
with open('hh_vacancies_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(data_dict, f, ensure_ascii=False, indent=2)